# Data Quality Check — before locking `world_bank_wide.csv` as the master dataset

Every module so far has been checked for its *own* step working correctly (row counts,
coverage %). This notebook checks the **cumulative result** after 5 merges — the kind of
issue that only shows up once everything's combined, not in any single module's own test.

Specifically checking for:
1. Merge artifacts (`_x`/`_y` suffixed duplicate columns from a re-run merge collision)
2. Correct dtypes (numeric columns that are secretly strings)
3. Structural integrity (still exactly one row per country-year, no drift)
4. Per-column missing-value rates, now that 5 sources are combined
5. Basic sanity ranges per column (catches unit/parsing errors, not just missingness)

In [12]:
import os
from pathlib import Path

while not (Path("config").exists() and Path("scripts").exists()):
    os.chdir("..")
    if Path.cwd() == Path.cwd().parent:
        raise RuntimeError("Could not locate project root")

print("Working directory set to:", os.getcwd())


Working directory set to: c:\Users\AJAY\Desktop\startup_implementation


In [15]:
import pandas as pd

df = pd.read_csv("data/cleaned/world_bank_wide.csv")
print("Shape:", df.shape)
print("Columns:", len(df.columns))


Shape: (4557, 41)
Columns: 41


## 1. Merge-artifact check

If any notebook was accidentally re-run against an already-merged file, pandas would create
`column_x`/`column_y` pairs instead of overwriting cleanly. This should come back empty.

In [16]:
suffix_cols = [c for c in df.columns if c.endswith("_x") or c.endswith("_y")]
dupe_cols = df.columns[df.columns.duplicated()].tolist()

if suffix_cols:
    print("PROBLEM — merge-suffix columns found:", suffix_cols)
else:
    print("Clean — no _x/_y merge-suffix columns.")

if dupe_cols:
    print("PROBLEM — literally duplicated column names:", dupe_cols)
else:
    print("Clean — no duplicate column names.")


Clean — no _x/_y merge-suffix columns.
Clean — no duplicate column names.


## 2. Structural integrity

Re-confirm one row per country-year, and that `country_code`/`year` themselves are clean.

In [17]:
dupes = df.duplicated(subset=["country_code", "year"]).sum()
print("Duplicate (country_code, year) rows:", dupes)

print("Distinct countries:", df["country_code"].nunique())
print("Year range:", df["year"].min(), "-", df["year"].max())
print("Expected max rows (countries x years):", df["country_code"].nunique() * df["year"].nunique())
print("Actual rows:", len(df))

blank_codes = df["country_code"].isna().sum() + (df["country_code"].astype(str).str.strip() == "").sum()
print("Blank/null country_code rows:", blank_codes)


Duplicate (country_code, year) rows: 0
Distinct countries: 217
Year range: 2005 - 2025
Expected max rows (countries x years): 4557
Actual rows: 4557
Blank/null country_code rows: 0


## 3. Data types

Every indicator column should be numeric (float/int). If World Bank, UNESCO, or WIPO ever
returned a value with stray formatting (a comma thousands-separator, a unit suffix, a quote
character), pandas would silently read that whole column as `object` (text) instead of
throwing an error — and every downstream stats function would either fail confusingly or
silently ignore the column.

In [18]:
non_numeric = df.drop(columns=["country_code", "country_name"], errors="ignore").select_dtypes(include="object").columns.tolist()

if non_numeric:
    print("PROBLEM — these should be numeric but are text/object dtype:", non_numeric)
    for c in non_numeric:
        print(f"\n{c} sample values:", df[c].dropna().unique()[:5])
else:
    print("Clean — all indicator columns are numeric dtype.")


Clean — all indicator columns are numeric dtype.


## 4. Missing-value rates per column

Not a pass/fail check by itself — sparsity is expected and real for several sources (UNESCO,
WIPO). This is about *knowing* the real numbers before you get to modeling, not being
surprised by them mid-regression.

In [19]:
missing = df.drop(columns=["country_code", "country_name", "year"], errors="ignore").isna().mean().sort_values(ascending=False)
missing_df = missing.reset_index()
missing_df.columns = ["variable", "pct_missing"]
missing_df["pct_missing"] = (missing_df["pct_missing"] * 100).round(1)
display(missing_df)


,variable,pct_missing
0,researchers_per_million,67.1
1,rd_expenditure,61.0
2,patent_applications,42.9
3,tertiary_enrollment,42.8
4,hightech_exports,41.5
5,education_expenditure,38.8
6,trademark_applications,38.7
7,new_business_density,36.1
8,new_businesses_registered,36.1
9,broad_money,32.0


## 5. Basic sanity ranges

Quick min/max per numeric column — not a substitute for real outlier analysis (that belongs
in the actual EDA module), just a fast way to catch obviously broken values now: negative
populations, percentages above 100 where they shouldn't be, a stray value 1000x too large from
a units mismatch, etc.

In [20]:
ranges = df.drop(columns=["country_code", "country_name", "year"], errors="ignore").agg(["min", "max"]).T
display(ranges)


,min,max
fdi_net_inflows,-3.434028e+11,7.338265e+11
renewable_energy,0.000000e+00,9.740000e+01
co2_per_capita,0.000000e+00,2.028652e+02
population_density,1.364917e-01,2.139344e+04
broad_money,4.298531e+00,1.330949e+04
inflation,-1.685969e+01,5.572018e+02
domestic_credit,4.976014e-01,3.010189e+02
rd_expenditure,5.490000e-03,6.345890e+00
control_of_corruption,-2.064721e+00,2.396962e+00
government_effectiveness,-2.242586e+00,2.292999e+00


## Verdict

If sections 1-3 all came back clean, `world_bank_wide.csv` is structurally sound and safe to
lock in as the master dataset. Section 4's missing-value rates and section 5's ranges are
worth glancing over for anything that looks obviously wrong (a max value 100x bigger than
its neighbors, a column that's 95%+ missing) — but genuine sparsity in UNESCO/WIPO columns is
expected, not a defect.

In [21]:
df.shape

(4557, 41)